# Comparing Classifiers Statistically

**Paper:** Alpaydın, E. (1999). *Combined 5×2cv F Test for Comparing Supervised Classification Learning Algorithms.* Neural Computation, 11(8), 1885–1892.

When you train two classifiers and one gets 87% and the other 85%, is that difference **real** or just noise from the random train/test split?

This notebook shows three statistical tests for answering that question:

| Test | When to use |
|------|-------------|
| **Alpaydın's 5×2cv F Test** | When you can retrain the gold standard |
| **McNemar's Test** | When you only have a fixed test set |
| **Paired t-test** | Common but inflated Type I error shown for comparison |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_breast_cancer, load_wine
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from neural_trees.statistical_tests import combined_5x2cv_f_test, mcnemar_test, paired_t_test

plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

## 1. The problem with naive comparison

Let's see how much accuracy can vary just from random splits — with no model change at all.

In [ ]:
X, y = load_iris(return_X_y=True)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# Same model, 20 different random splits
accs = []
for seed in range(20):
    X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.2, random_state=seed)
    clf = DecisionTreeClassifier(max_depth=4, random_state=42)
    clf.fit(X_tr, y_tr)
    accs.append(clf.score(X_te, y_te))

accs = np.array(accs)
print(f'Same model, 20 random splits:')
print(f'  Min: {accs.min():.3f}  Max: {accs.max():.3f}')
print(f'  Std: {accs.std():.3f}')
print(f'  Range: {accs.max() - accs.min():.3f}')
print()
print('→ Accuracy can swing by this much just from the split.')
print('→ A difference smaller than this is meaningless.')

plt.figure(figsize=(9, 4))
plt.bar(range(20), accs, color='#667eea', alpha=0.8)
plt.axhline(accs.mean(), color='red', linestyle='--', label=f'Mean: {accs.mean():.3f}')
plt.xlabel('Random seed')
plt.ylabel('Test accuracy')
plt.title('Same model, 20 random train/test splits', fontsize=13)
plt.ylim(0.8, 1.0)
plt.legend()
plt.tight_layout()
plt.show()

## 2. Alpaydın's Combined 5×2cv F Test

**How it works:**
- Repeat 2-fold CV 5 times (10 difference measurements)
- Estimate variance within each repetition
- Compute F statistic — follows F(10, 5) distribution under H0

This avoids the inflated Type I error of the paired t-test.

In [ ]:
X, y = load_iris(return_X_y=True)
scaler = StandardScaler()
X_s = scaler.fit_transform(X)

pairs = [
    ('Decision Tree', DecisionTreeClassifier(random_state=42),
     'Random Forest', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42),
     'SVM (RBF)',     SVC(kernel='rbf', probability=True, random_state=42)),
    ('Decision Tree', DecisionTreeClassifier(random_state=42),
     'Dummy',         DummyClassifier(strategy='most_frequent')),
]

print(f'{"Comparison":<35} {"F-stat":>8} {"p-value":>8} {"Decision"}')
print('-' * 75)

for name_a, clf_a, name_b, clf_b in pairs:
    result = combined_5x2cv_f_test(clf_a, clf_b, X_s, y)
    decision = 'REJECT H0 ✓' if result.reject_null else 'no difference'
    print(f'{name_a} vs {name_b:<20} {result.statistic:>8.3f} {result.p_value:>8.4f}   {decision}')

## 3. McNemar's Test — when you have a fixed test set

In [ ]:
X, y = load_iris(return_X_y=True)
X_s = StandardScaler().fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.3, random_state=42)

clf_a = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)
clf_b = SVC(kernel='rbf', random_state=42).fit(X_tr, y_tr)

pred_a = clf_a.predict(X_te)
pred_b = clf_b.predict(X_te)

result = mcnemar_test(y_te, pred_a, pred_b)
print(result)
print()
print(f'DT  accuracy: {(pred_a == y_te).mean():.4f}')
print(f'SVM accuracy: {(pred_b == y_te).mean():.4f}')

## 4. Comparing all three tests side by side

In [ ]:
X, y = load_iris(return_X_y=True)
X_s = StandardScaler().fit_transform(X)
X_tr, X_te, y_tr, y_te = train_test_split(X_s, y, test_size=0.3, random_state=42)

clf_a = DecisionTreeClassifier(max_depth=4, random_state=42)
clf_b = KNeighborsClassifier(n_neighbors=5)

clf_a.fit(X_tr, y_tr)
clf_b.fit(X_tr, y_tr)

r_f    = combined_5x2cv_f_test(clf_a, clf_b, X_s, y)
r_mc   = mcnemar_test(y_te, clf_a.predict(X_te), clf_b.predict(X_te))
r_t    = paired_t_test(clf_a, clf_b, X_s, y)

print('Decision Tree vs KNN three tests')
print('=' * 50)
for r in [r_f, r_mc, r_t]:
    print(f'  {r.test_name}')
    print(f'    p-value  = {r.p_value:.4f}')
    print(f'    decision = {"REJECT H0" if r.reject_null else "fail to reject H0"}')
    print()

## 5. When tests disagree why?

The paired t-test reuses training data across folds. The same samples appear in multiple
training sets, making the differences correlated. This inflates the false positive rate.

Alpaydın's F test estimates variance *within* each 2-fold repetition, correcting for this.

**Rule of thumb:** If in doubt, use the 5×2cv F test. It's the most conservative and best calibrated.